## How to run in Colab
1. Open this notebook on GitHub and click **Open in Colab** (or go to colab.research.google.com → File → Open notebook → GitHub and paste `simonlivson/crypto-market-explorer`).
2. Click **Runtime → Run all**.
3. I don't use any API key. If the CoinGecko or Coin Metrics API is down or rate-limited, my code automatically loads the saved CSV files from my GitHub repo instead, so every cell still runs.

# Crypto Market Explorer: Cycles, Altseasons and Coins That Break Away From Bitcoin

Most of the crypto market moves together with Bitcoin. But in some phases altcoins outperform (an "altseason"), and some coins break away from Bitcoin for months, like Hyperliquid and Zcash in 2025.

In this project I want to find out in which market phases diversification in crypto is real, and which coins are just "Bitcoin in disguise". My story has two parts:

- **Part A, the big picture:** Bitcoin and Ethereum daily history since 2013/2015, to study the 4-year cycle and altseasons.
- **Part B, inside the current phase:** the last 365 days for 12 coins, to see who follows Bitcoin and who breaks away.

Before I can analyze anything, I need data. This first section collects it.

# 1. Data Collection

In this section I collect three datasets:

| Dataset | Source | What it gives me |
|---|---|---|
| `coins_365d.csv` | CoinGecko API (free, no key) | daily price, market cap and volume for 12 coins, last 365 days |
| `btc_eth_history.csv` | Coin Metrics Community API (free, no key) | daily BTC and ETH prices since 2013 / 2015 |
| `coin_categories.csv` | my own file | the theme of each coin (Layer 1, Privacy, AI, ...) |

I save the raw API answers as JSON files and the cleaned tables as CSV files in my `data/` folder. If an API fails, I fall back to the CSV copies on GitHub.

## 1.1 Importing my libraries

I only need a few libraries here:
- `requests` sends HTTP requests to the APIs (like a browser does when it opens a page).
- `pandas` turns the answers into tables (DataFrames).
- `json` and `os` let me save the raw files to my folders.
- `time` lets me pause between requests so the free API doesn't block me.

In [ ]:
import os
import json
import time

import requests
import pandas as pd

# my folders for the raw API answers and the CSV files
os.makedirs("data/raw", exist_ok=True)

# backup copies of my data on GitHub, in case an API fails
GITHUB_DATA = "https://raw.githubusercontent.com/simonlivson/crypto-market-explorer/main/data/"


## 1.2 Choosing my 12 coins

CoinGecko doesn't use tickers like "BTC". Every coin has its own **id** (for example `binancecoin` for BNB and `ripple` for XRP). I checked each id on CoinGecko, and I list them here with the ticker I'll use in my charts.

I picked large Layer 1 coins (BTC, ETH, SOL, ADA), BNB, XRP for payments, DOGE as a meme coin, LINK as an oracle, and four "theme" coins that had their own runs: HYPE (DEX/perps), ZEC and XMR (privacy) and TAO (AI).

In [ ]:
# CoinGecko id -> ticker
coins = {
    "bitcoin": "BTC",
    "ethereum": "ETH",
    "solana": "SOL",
    "binancecoin": "BNB",
    "ripple": "XRP",
    "dogecoin": "DOGE",
    "cardano": "ADA",
    "chainlink": "LINK",
    "hyperliquid": "HYPE",
    "zcash": "ZEC",
    "monero": "XMR",
    "bittensor": "TAO",
}
print(len(coins), "coins")


## 1.3 A safe way to call the API

The free CoinGecko API allows only a limited number of requests per minute. If I ask too fast, it answers with status code **429 (Too Many Requests)**.

So I write a small helper function `get_json()`:
- it tries the request up to 4 times,
- if the answer is 429 or a server error, it waits a bit longer each time (10, 20, 30 seconds) and tries again,
- if everything fails, it returns `None`, so my notebook doesn't crash.

In [ ]:
def get_json(url, params=None, tries=4):
    for attempt in range(1, tries + 1):
        try:
            response = requests.get(url, params=params, timeout=30)
            # 200 means everything is fine
            if response.status_code == 200:
                return response.json()
            print(f"  status {response.status_code}, attempt {attempt}")
        except requests.RequestException as error:
            print(f"  error: {error}, attempt {attempt}")
        # I wait longer after every failed try
        time.sleep(10 * attempt)
    return None


## 1.4 Fetching 365 days of data for each coin

For each coin I call the `market_chart` endpoint:

`https://api.coingecko.com/api/v3/coins/{id}/market_chart?vs_currency=usd&days=365&interval=daily`

The answer is JSON with three lists: `prices`, `market_caps` and `total_volumes`. Each item is a pair `[timestamp, value]`, where the timestamp is in milliseconds since 1970.

I save every answer as it is in `data/raw/<coin>.json`, so I always keep the original data. Between coins I pause 3 seconds to stay polite to the free API.

In [ ]:
url = "https://api.coingecko.com/api/v3/coins/{}/market_chart"
params = {"vs_currency": "usd", "days": 365, "interval": "daily"}

raw_data = {}
for coin_id in coins:
    print("fetching", coin_id)
    data = get_json(url.format(coin_id), params)
    if data is None:
        print("  failed:", coin_id)
        continue
    raw_data[coin_id] = data
    # I keep the original answer as a JSON file
    with open(f"data/raw/{coin_id}.json", "w") as f:
        json.dump(data, f)
    time.sleep(3)

print(f"I got data for {len(raw_data)} of {len(coins)} coins")


## 1.5 Turning the JSON into one table

Now I turn the three lists of every coin into one **long-format** table. Long format means one row per coin per day, with these columns:

`date, coin, price, market_cap, volume`

This format is easy to filter (`df[df.coin == "BTC"]`), group (`groupby("coin")`) and pivot later.

Two small details:
- I convert the millisecond timestamps to dates with `pd.to_datetime(..., unit="ms")`.
- CoinGecko adds the current, unfinished day as the last point. I keep only one row per day and drop that last partial day, so every day is a complete one.

In [ ]:
tables = []
for coin_id, data in raw_data.items():
    # one small table per coin, one column per list
    prices = pd.DataFrame(data["prices"], columns=["ts", "price"])
    caps = pd.DataFrame(data["market_caps"], columns=["ts", "market_cap"])
    volumes = pd.DataFrame(data["total_volumes"], columns=["ts", "volume"])

    coin_df = prices.merge(caps, on="ts").merge(volumes, on="ts")
    coin_df["date"] = pd.to_datetime(coin_df["ts"], unit="ms").dt.normalize()
    coin_df["coin"] = coins[coin_id]
    tables.append(coin_df)

if tables:
    coins_df = pd.concat(tables, ignore_index=True)
    # one row per coin and day
    coins_df = coins_df.drop_duplicates(subset=["coin", "date"], keep="first")
    # the last day is still running, so I drop it
    coins_df = coins_df[coins_df["date"] < pd.Timestamp.today().normalize()]
    coins_df = coins_df[["date", "coin", "price", "market_cap", "volume"]]
else:
    coins_df = pd.DataFrame()

# if the API gave me nothing (or only part), I load my saved copy from GitHub
if len(raw_data) < len(coins):
    print("API failed, loading the saved CSV from GitHub")
    coins_df = pd.read_csv(GITHUB_DATA + "coins_365d.csv", parse_dates=["date"])
else:
    coins_df.to_csv("data/coins_365d.csv", index=False)

coins_df.head()


### A first check of my coin data

I check how many rows each coin has and which dates they cover. A coin that is younger than one year (or listed later on CoinGecko) will have fewer rows. That matters later, because I can only compare coins over the days they all have.

In [ ]:
summary = coins_df.groupby("coin")["date"].agg(["count", "min", "max"])
summary.columns = ["days", "first_date", "last_date"]
print("total rows:", len(coins_df))
summary


## 1.6 Long-term Bitcoin and Ethereum history

CoinGecko's free tier only gives me 365 days of daily data, and one year is not enough to see a 4-year cycle. So for Part A I use the **Coin Metrics Community API**, which is free and needs no key:

`https://community-api.coinmetrics.io/v4/timeseries/asset-metrics?assets=btc,eth&metrics=PriceUSD&frequency=1d`

It returns a daily USD reference price for BTC (from 2010) and ETH (from August 2015). I start in 2013, as planned.

The answer comes in **pages**. Each page has a `next_page_url` if there's more data, so I keep following that link until there are no pages left.

In [ ]:
cm_url = "https://community-api.coinmetrics.io/v4/timeseries/asset-metrics"
cm_params = {
    "assets": "btc,eth",
    "metrics": "PriceUSD",
    "frequency": "1d",
    "start_time": "2013-01-01",
    "page_size": 10000,
}

rows = []
page = get_json(cm_url, cm_params)
while page is not None:
    rows.extend(page["data"])
    # I follow the next page until there is none
    next_url = page.get("next_page_url")
    if not next_url:
        break
    page = get_json(next_url)

if rows:
    with open("data/raw/coinmetrics_btc_eth.json", "w") as f:
        json.dump(rows, f)
    history = pd.DataFrame(rows)
    history["date"] = pd.to_datetime(history["time"]).dt.tz_localize(None).dt.normalize()
    history["price"] = history["PriceUSD"].astype(float)
    # one column per coin: date, BTC, ETH
    history = history.pivot_table(index="date", columns="asset", values="price").reset_index()
    history.columns.name = None
    history = history.rename(columns={"btc": "BTC", "eth": "ETH"})
    history.to_csv("data/btc_eth_history.csv", index=False)
else:
    print("API failed, loading the saved CSV from GitHub")
    history = pd.read_csv(GITHUB_DATA + "btc_eth_history.csv", parse_dates=["date"])

print(len(history), "days from", history["date"].min().date(), "to", history["date"].max().date())
print("first ETH price on", history.loc[history["ETH"].notna(), "date"].min().date())
history.tail()


Ethereum didn't exist before 2015, so its column is empty (NaN) for the early years. That's expected, not an error. I'll handle it in the cleaning section.

## 1.7 My coin categories

The last dataset is a small CSV I wrote by hand. It gives each coin a **theme**, so later I can check whether coins that break away from Bitcoin share a theme (hypothesis 7). I'll merge it with my price data using the `symbol` column.

In [ ]:
try:
    categories = pd.read_csv("data/coin_categories.csv")
except FileNotFoundError:
    # in Colab the file isn't local, so I read it from GitHub
    categories = pd.read_csv(GITHUB_DATA + "coin_categories.csv")
categories


## 1.8 Summary of my data collection

I now have three datasets:
1. **coins_365d.csv**: 12 coins × about 365 days of price, market cap and volume from CoinGecko (raw JSON kept in `data/raw/`).
2. **btc_eth_history.csv**: daily BTC prices since 2013 and ETH since 2015 from Coin Metrics.
3. **coin_categories.csv**: the theme of every coin.

Next, in section 2, I clean the data: missing values, duplicates, data types and outliers.